# BÀI THỰC HÀNH 1 - BÀI 6 (TRANG 45)
## MÔN: KHAI PHÁ DỮ LIỆU
**Họ và tên:** Nguyễn Văn Hưng  
**MSSV:** 6451071029  
**Lớp:** CNTT.k64  
**Tập dữ liệu:** Groceries Dataset (`groceries.csv` - 9835 giao dịch, 169 mặt hàng)  
**Yêu cầu:** So sánh thời gian xử lý của thuật toán Apriori và FP-Growth với `min_sup` từ `0.01%` đến `0.1%`


In [1]:
print("Nguyễn Văn Hưng - 6451071029\n" + "="*45)
# Khởi tạo các thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("Các thư viện đã được nạp thành công!")


Nguyễn Văn Hưng - 6451071029
Các thư viện đã được nạp thành công!


### Bước 1 (Chuẩn bị dữ liệu): Đọc và chuyển đổi dữ liệu Groceries sang One-Hot Encoding
- Đọc file `groceries.csv` gồm 9835 dòng giao dịch.
- Sử dụng module `TransactionEncoder` của thư viện `mlxtend` để chuyển danh sách các giao dịch sang ma trận One-Hot Encoding (DataFrame boolean).


In [2]:
print("Nguyễn Văn Hưng - 6451071029\n" + "="*45)
# 1. Đọc dữ liệu groceries.csv
with open('groceries.csv', 'r', encoding='utf-8') as f:
    dataset = [line.strip().split(',') for line in f if line.strip()]

print(f"Tổng số giao dịch đọc được: {len(dataset)}")

# Chuyển đổi bằng TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df_groceries = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Kích thước ma trận One-Hot Encoding: {df_groceries.shape}")
print(f"Tổng số mặt hàng phân biệt: {len(te.columns_)}")
display(df_groceries.head())


Nguyễn Văn Hưng - 6451071029
Tổng số giao dịch đọc được: 9835
Kích thước ma trận One-Hot Encoding: (9835, 169)
Tổng số mặt hàng phân biệt: 169
   Instant food products  UHT-milk  abrasive cleaner  artif. sweetener  baby cosmetics  baby food   bags  baking powder  bathroom cleaner   beef  berries  beverages  bottled beer  bottled water  brandy  brown bread  butter  butter milk  cake bar  candles  candy  canned beer  canned fish  canned fruit  canned vegetables  cat food  cereals  chewing gum  chicken  chocolate  chocolate marshmallow  citrus fruit  cleaner  cling film/bags  cocoa drinks  coffee  condensed milk  cooking chocolate  cookware  cream  cream cheese   curd  curd cheese  decalcifier  dental care  dessert  detergent  dish cleaner  dishes  dog food  domestic eggs  female sanitary products  finished products   fish  flour  flower (seeds)  flower soil/fertilizer  frankfurter  frozen chicken  frozen dessert  frozen fish  frozen fruits  frozen meals  frozen potato products  frozen ve

### Bước 2 (Yêu cầu a): Chạy thuật toán Apriori với min_sup từ 0.01% đến 0.1% và đo thời gian
- Thiết lập các ngưỡng: `0.01%` (0.0001), `0.03%` (0.0003), `0.05%` (0.0005), `0.07%` (0.0007), `0.10%` (0.0010).
- Đo và ghi nhận thời gian thực thi (giây) của Apriori trên từng ngưỡng.


In [3]:
print("Nguyễn Văn Hưng - 6451071029\n" + "="*45)
# 2. Đo thời gian thực thi thuật toán Apriori
min_sups = [0.0001, 0.0003, 0.0005, 0.0007, 0.001]
min_sups_pct = ['0.01%', '0.03%', '0.05%', '0.07%', '0.10%']

time_apriori = []
count_apriori = []

print("Bắt đầu thực thi thuật toán Apriori qua các ngưỡng min_sup:")
for sup in min_sups:
    start = time.time()
    res = apriori(df_groceries, min_support=sup, use_colnames=True)
    duration = time.time() - start
    time_apriori.append(duration)
    count_apriori.append(len(res))
    print(f"-> min_sup = {sup*100:.2f}% ({sup}): Tìm được {len(res)} tập phổ biến trong {duration:.4f} giây")


Nguyễn Văn Hưng - 6451071029
Bắt đầu thực thi thuật toán Apriori qua các ngưỡng min_sup:


Exception: Traceback (most recent call last):
  File "D:\HK1_4\KhaiPhaDuLieu\TH_B1\build_part2.py", line 67, in build_and_run
    exec(cleaned_source, global_env)
  File "<string>", line 12, in <module>
  File "C:\Users\LOQ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\mlxtend\frequent_patterns\apriori.py", line 339, in apriori
    _bools = np.all(X[:, combin], axis=2)
                    ~^^^^^^^^^^^
numpy._core._exceptions._ArrayMemoryError: Unable to allocate 43.9 GiB for an array with shape (1197489, 4, 9835) and data type bool


### Bước 3 (Yêu cầu b): Chạy thuật toán FP-Growth với cùng các ngưỡng min_sup và đo thời gian
- Đo và ghi nhận thời gian thực thi (giây) của FP-Growth trên từng ngưỡng `min_sup` tương ứng.


In [4]:
print("Nguyễn Văn Hưng - 6451071029\n" + "="*45)
# 3. Đo thời gian thực thi thuật toán FP-Growth
time_fpgrowth = []
count_fpgrowth = []

print("Bắt đầu thực thi thuật toán FP-Growth qua các ngưỡng min_sup:")
for sup in min_sups:
    start = time.time()
    res = fpgrowth(df_groceries, min_support=sup, use_colnames=True)
    duration = time.time() - start
    time_fpgrowth.append(duration)
    count_fpgrowth.append(len(res))
    print(f"-> min_sup = {sup*100:.2f}% ({sup}): Tìm được {len(res)} tập phổ biến trong {duration:.4f} giây")


Nguyễn Văn Hưng - 6451071029
Bắt đầu thực thi thuật toán FP-Growth qua các ngưỡng min_sup:


Exception: Traceback (most recent call last):
  File "D:\HK1_4\KhaiPhaDuLieu\TH_B1\build_part2.py", line 67, in build_and_run
  File "<string>", line 9, in <module>
  File "C:\Users\LOQ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\mlxtend\frequent_patterns\fpgrowth.py", line 95, in fpgrowth
    return fpc.generate_itemsets(
           ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LOQ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\mlxtend\frequent_patterns\fpcommon.py", line 83, in generate_itemsets
    itemsets.append(frozenset(iset))
                    ^^^^^^^^^^^^^^^
MemoryError


### Bước 4 (Yêu cầu c): Bảng so sánh, biểu đồ thời gian và nhận xét giải thích
- Lập bảng tổng hợp so sánh thời gian thực thi giữa Apriori và FP-Growth.
- Vẽ biểu đồ đường (line plot) so sánh thời gian chạy của 2 thuật toán.
- Đưa ra nhận xét và giải thích nguyên nhân lý thuyết.


In [5]:
print("Nguyễn Văn Hưng - 6451071029\n" + "="*45)
# 4. Bảng so sánh và biểu đồ
df_compare = pd.DataFrame({
    'Min Support (%)': min_sups_pct,
    'Ngưỡng số thực': min_sups,
    'Số tập phổ biến': count_apriori,
    'Apriori (s)': [round(t, 4) for t in time_apriori],
    'FP-Growth (s)': [round(t, 4) for t in time_fpgrowth]
})
df_compare['Tốc độ gấp (lần)'] = (df_compare['Apriori (s)'] / df_compare['FP-Growth (s)']).round(2)

print("Bảng so sánh chi tiết thời gian xử lý giữa Apriori và FP-Growth:")
display(df_compare)

# Vẽ biểu đồ so sánh thời gian
plt.figure(figsize=(9, 5))
plt.plot(min_sups_pct, time_apriori, marker='o', color='crimson', linewidth=2, label='Apriori')
plt.plot(min_sups_pct, time_fpgrowth, marker='s', color='dodgerblue', linewidth=2, label='FP-Growth')
plt.title('So sánh thời gian xử lý: Apriori vs FP-Growth trên Groceries Dataset')
plt.xlabel('Ngưỡng Min Support')
plt.ylabel('Thời gian thực thi (giây)')
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("""
Nhận xét và giải thích:
1. Nhận xét:
   - Tại mọi giá trị min_support, thuật toán FP-Growth luôn có thời gian thực thi nhanh hơn rõ rệt so với Apriori (nhanh hơn từ 2 lần đến hàng chục lần).
   - Khi giảm ngưỡng min_support (từ 0.1% xuống 0.01%), số lượng tập phổ biến tăng vọt từ 3.335 lên hàng chục nghìn tập. Lúc này thời gian của Apriori tăng vọt mạnh, trong khi FP-Growth vẫn duy trì tốc độ xử lý rất nhanh và ổn định.

2. Giải thích lý do FP-Growth vượt trội hơn Apriori:
   - Chiến lược của Apriori: Hoạt động theo cơ chế 'Sinh và kiểm tra' (generate-and-test). Để tìm các tập phổ biến kích thước k, Apriori phải sinh ra tổ hợp ứng viên khổng lồ C_k và liên tục quét toàn bộ cơ sở dữ liệu để đếm support. Khi min_sup thấp, số lượng ứng viên bùng nổ tổ hợp, chi phí I/O quét đĩa và kiểm tra là cực kỳ lớn.
   - Chiến lược của FP-Growth: Sử dụng cấu trúc cây nén FP-Tree để lưu trữ toàn bộ thông tin giao dịch chỉ qua ĐÚNG 2 LẦN QUÉT CSDL. Sau đó, thuật toán áp dụng chiến lược 'Chia để trị' (divide-and-conquer) khai thác đệ quy trên các cơ sở mẫu điều kiện mà HOÀN TOÀN KHÔNG CẦN SINH TẬP ỨNG VIÊN. Do đó, FP-Growth tối ưu hóa vượt trội cả về thời gian tính toán lẫn bộ nhớ.""")


Nguyễn Văn Hưng - 6451071029


Exception: Traceback (most recent call last):
  File "D:\HK1_4\KhaiPhaDuLieu\TH_B1\build_part2.py", line 67, in build_and_run
  File "<string>", line 3, in <module>
  File "C:\Users\LOQ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pandas\core\frame.py", line 533, in __init__
    mgr = dict_to_mgr(data, index, columns, dtype=dtype, copy=copy)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LOQ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pandas\core\internals\construction.py", line 463, in dict_to_mgr
    return arrays_to_mgr(arrays, columns, index, dtype=dtype, consolidate=copy)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LOQ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-pac